# From Python Beginner to Cloud Automation
## Python, PowerShell, Microsoft 365, and Azure with JupyterLab

A practical, beginner-friendly course inspired by examples in [ScriptLab](https://github.com/steble70/scriptlab). The course starts with Python and JupyterLab, then introduces PowerShell and cloud administration concepts.

> **How to use this notebook:** Work from top to bottom and run Python cells with **Shift+Enter**. PowerShell examples are shown as text and should be run in PowerShell 7, not a Python cell, unless specifically noted. Cloud exercises are optional and require an authorized test environment.

> **Safety:** Cloud examples are read-only by default. Never commit passwords, tokens, private keys, tenant data, or sensitive device information to a notebook or Git repository.

## Chapters
1. JupyterLab, notebooks, and a safe workspace
2. Python basics: values, variables, and expressions
3. Decisions, loops, and collections
4. Functions and reusable scripts
5. Files, paths, CSV, and JSON
6. Testing, errors, and debugging
7. PowerShell fundamentals and the object pipeline
8. PowerShell functions and system inventory
9. Cloud concepts and identity
10. Azure: connect and inspect resources
11. Microsoft 365: Microsoft Graph and read-only queries
12. Security, permissions, and responsible automation
13. Capstone: a small cloud-inventory report

Examples and terminology are based on ScriptLab materials, including `JUTU.ipynb`, `minicalc.ipynb`, `ccprep.py`, `gn.ps1`, the PowerShell profiles, and the AZ-900/MS-102 glossaries.

## 1. JupyterLab, notebooks, and a safe workspace

JupyterLab is an interactive environment for combining explanatory Markdown, executable code, and results. A notebook has a Python kernel by default. Keep project files together, use relative paths, and avoid placing credentials or private data in the project.

The repository's `JUTU.ipynb` introduces notebooks, cells, and project folders. This example creates a local folder only; it does not change cloud resources.

In [ ]:
from pathlib import Path

project_dir = Path.cwd() / "tutorial-data"
project_dir.mkdir(exist_ok=True)
print(f"Working folder: {project_dir}")

**Try it:** Print the current working directory with `Path.cwd()`. What changes if you open the notebook from a different folder?

**You learned:** A notebook combines narrative and code, and `pathlib` offers a portable way to work with file paths.

## 2. Python basics: values, variables, and expressions

A variable gives a name to a value. Python commonly uses strings, integers, floats, and booleans. The repository's arithmetic and mini-calculator exercises are good starting points for expressions and output formatting.

In [ ]:
course_name = "ScriptLab"
lessons_completed = 3
completion_rate = 0.25

print(f"{course_name}: {lessons_completed} lessons completed")
print(f"Progress: {completion_rate:.0%}")

**Try it:** Change the variables and display the result of a simple calculation. Convert a string containing a number to an integer with `int()`.

**You learned:** Variables hold values, and f-strings make it easy to format output.

## 3. Decisions, loops, and collections

Use `if` to choose between actions and loops to repeat work. Lists hold ordered values; dictionaries associate keys with values. These patterns are useful for quizzes and practice generators.

In [ ]:
scores = [8, 6, 10, 7]
passed = [score for score in scores if score >= 7]

print("Passing scores:", passed)
print(f"Average: {sum(scores) / len(scores):.1f}")

**Try it:** Write a loop that labels each score as `pass` or `retry`. What should happen if the list is empty?

**You learned:** Conditions and loops control flow; collections let you work with groups of related values.

## 4. Functions and reusable scripts

Functions name a reusable unit of work. Parameters provide input and `return` sends a result back. Type hints and docstrings help explain intent. Later, code can be moved from a notebook into a `.py` module, as in the repository's `ccprep.py`.

In [ ]:
def greeting(name: str) -> str:
    """Return a short greeting for name."""
    return f"Hello, {name}!"

greeting("learner")

**Try it:** Add a second parameter for a preferred language and return a greeting in that language.

**You learned:** Functions make code easier to reuse, explain, and test.

## 5. Files, paths, CSV, and JSON

JSON is a common format for configuration and structured data. ScriptLab includes AZ-900, DP-900, and MS-102 glossary JSON files. The example below loads the AZ-900 glossary if the file is in the current folder. It deliberately handles the case where the file is not present.

In [ ]:
import json
from pathlib import Path

glossary_path = Path("AZ-900_glossary.json")
if glossary_path.exists():
    glossary_data = json.loads(glossary_path.read_text(encoding="utf-8"))
    glossary = glossary_data["Glossary"]
    print(list(glossary.items())[:3])
else:
    glossary = {}
    print("Place AZ-900_glossary.json in the notebook's working folder to load it.")

The repository's `ccprep.py` is a useful code-review example: a quiz should choose a term and its definition together, rather than choosing a key and value independently.

In [ ]:
import random

if glossary:
    term, definition = random.choice(list(glossary.items()))
    print(f"What does {term!r} mean?\n\nAnswer: {definition}")
else:
    print("Load the glossary first to run the quiz example.")

**Try it:** Write the loaded glossary to a new JSON file in `project_dir`. For CSV practice, use Python's built-in `csv` module or the repository's small CSV reference files.

**You learned:** Use explicit encodings and `pathlib`; keep related JSON keys and values together.

## 6. Testing, errors, and debugging

An exception reports a problem; validation can make the reason clear. Assertions are useful for quick checks in a tutorial. Avoid silently suppressing errors when you need to understand why an operation failed.

In [ ]:
def safe_average(values: list[float]) -> float:
    """Return the mean of a non-empty list of numbers."""
    if not values:
        raise ValueError("Provide at least one value.")
    return sum(values) / len(values)

assert safe_average([2, 4]) == 3
safe_average([2, 4, 6])

**Try it:** Call `safe_average([])` and read the error. Add a test for a single value.

**You learned:** Validate assumptions and use errors to make failures understandable.

## 7. PowerShell fundamentals and the object pipeline

PowerShell commands (cmdlets) commonly use `Verb-Noun` names and pass objects—not just text—through the pipeline. `Where-Object` filters objects and `Select-Object` chooses properties. This example inspects local processes; run it in PowerShell 7, not the Python kernel. The repository's `admincmdlets.csv` contains additional command examples.

```powershell
Get-Process |
    Where-Object CPU -GT 10 |
    Select-Object ProcessName, CPU |
    Sort-Object CPU -Descending
```

**Try it:** Remove the filter and select a different property. Consider why a process listing may vary between computers.

**You learned:** PowerShell's object pipeline supports filtering, projection, and sorting.

## 8. PowerShell functions and system inventory

The repository's `gn.ps1` demonstrates functions and returning a `PSCustomObject` assembled from CIM data. Start with a low-impact, read-only query such as the operating system information below. Do not share inventory containing serial numbers, IP addresses, hostnames, or other identifying details. Some existing scripts request administrator privileges or access sensitive device information; read and understand a script before running it.

```powershell
Get-CimInstance -ClassName Win32_OperatingSystem |
    Select-Object Caption, Version, LastBootUpTime
```

A function can package this query for reuse and return structured output.

```powershell
function Get-OsSummary {
    Get-CimInstance -ClassName Win32_OperatingSystem |
        Select-Object Caption, Version, LastBootUpTime
}
Get-OsSummary
```

**Try it:** Add a parameter to the function that controls which computer is queried, then consider what permissions and network configuration that requires.

## 9. Cloud concepts and identity

Use the repository's AZ-900 and MS-102 glossaries to explore cloud vocabulary. Key ideas include cloud computing, subscriptions, resource groups, regions, IaaS/PaaS/SaaS, tenants, identity, MFA, Conditional Access, RBAC, and least privilege.

A **subscription** is a billing and management boundary for Azure resources; a **resource group** is a logical container for related resources. Microsoft Entra ID is the current name for Azure Active Directory (Azure AD). Microsoft 365 and Azure share identity concepts, but the permissions and management surfaces for services differ.

**Exercise:** Choose three terms from `AZ-900_glossary.json` or `MS-102_glossary.json`. Explain each in your own words and identify an example of how it affects an administrator's work.

## 10. Azure: connect and inspect resources

Azure PowerShell examples require the Az module, an Azure sign-in, and permission to view resources in a subscription. Start with inspection. Do not run create, start, stop, update, or delete commands unless you understand their effects, have authorization, and are using an approved test environment. Azure resources can incur charges.

```powershell
# Install the Az module only if needed, following your organization's guidance.
# Install-Module Az -Scope CurrentUser
Connect-AzAccount
Get-AzContext
Get-AzVM | Select-Object Name, ResourceGroupName, Location
```

The repository's `admincmdlets.csv` includes Azure command names such as `Get-AzVM` and `New-AzVM`. Treat command references as material to study, not instructions to execute blindly. Never save access tokens, credentials, subscription identifiers, or sensitive output in a committed notebook.

**No Azure access?** Use a small Python list of fictional resources and practice filtering it instead.

**Try it:** Explain the difference between signing in, selecting a subscription context, and having permission to read a resource.

## 11. Microsoft 365: Microsoft Graph and read-only queries

Microsoft Graph provides API access to Microsoft 365 data. The Microsoft Graph PowerShell SDK and permissions must be configured for your environment. The example below is illustrative and may require module installation, tenant consent, and an authorized account. Request only the permissions needed; do not use a production tenant for experiments without approval.

```powershell
# Example only: use an authorized test tenant and approved permissions.
Connect-MgGraph -Scopes "User.Read.All"
Get-MgUser -Top 5 | Select-Object DisplayName, UserPrincipalName
```

The Microsoft 365 glossary topics in the repository include tenants, Exchange Online, SharePoint Online, Teams, MFA, Conditional Access, Microsoft Graph, and RBAC. A glossary entry explains a concept; it does not grant authorization or ensure a command is suitable for your tenant.

**No tenant access?** Practice with fictional user records in Python. Use no real names, email addresses, or tenant information.

## 12. Security, permissions, and responsible automation

Use least privilege: request only the access needed for the task, and prefer read-only permissions while learning. MFA, Conditional Access, RBAC, data protection, and auditability are recurring topics in the repository's Microsoft 365 and security glossaries.

- Never put a password, API key, access token, or private key in a notebook or source file.
- Do not commit tenant data, personal data, device inventory, or unredacted command output.
- Review commands and their effects before execution; distinguish queries from changes.
- Use approved test environments, follow organizational policy, and ask an administrator when permission or impact is unclear.
- Keep example data fictional and sanitize outputs before sharing.

**Exercise:** Review one command from `admincmdlets.csv`. Label it read-only or state-changing, list the permissions it might need, and describe a safe way to test it.

## 13. Capstone: a small cloud-inventory report

Build a report using mock data first. If you have an authorized test tenant and subscription, you may replace the mock data with approved, read-only queries. Keep the report sanitized and do not include personal or tenant identifiers.

### Goals
1. Load a glossary JSON file with Python.
2. Create a small Python function that validates and summarizes a list of fictional resources.
3. Optionally collect read-only Azure or Microsoft 365 results using PowerShell in a separate terminal.
4. Save only fictional or sanitized data to CSV.
5. Document permissions, data-safety decisions, and how to reproduce the work.

### Mock-data example
The following sample is entirely fictional and has no external dependencies.

In [ ]:
import csv

mock_resources = [
    {"name": "demo-storage-01", "kind": "Storage", "region": "example-region"},
    {"name": "demo-app-01", "kind": "App", "region": "example-region"},
]

def summarize_resources(resources: list[dict[str, str]]) -> dict[str, int]:
    """Count fictional resources by kind."""
    counts: dict[str, int] = {}
    for resource in resources:
        kind = resource["kind"]
        counts[kind] = counts.get(kind, 0) + 1
    return counts

summary = summarize_resources(mock_resources)
print(summary)

report_path = project_dir / "mock-resource-report.csv"
with report_path.open("w", newline="", encoding="utf-8") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=["name", "kind", "region"])
    writer.writeheader()
    writer.writerows(mock_resources)

print(f"Wrote fictional example report to {report_path}")

### Capstone checklist
- Does the notebook run from top to bottom using mock data?
- Are functions documented and tested with suitable inputs?
- Are optional cloud commands clearly separated from mock-data examples?
- Are permissions and possible costs explained?
- Have all secrets and identifying data been excluded from outputs?

## Next steps
Continue by exploring the original notebooks and scripts in ScriptLab. Extend one exercise at a time, test changes with mock data, and document what you learn.